In [ ]:
%cd ../../

In [ ]:
import sys
import datetime
from pathlib import Path

import lightning as L
import torch
import yaml
import polars as pl
from torch import Tensor, nn
from torch.nn import Module
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from lightning.pytorch.callbacks import RichProgressBar, LearningRateMonitor

from lightning.pytorch.loggers import TensorBoardLogger
from torch.nn import functional as F
from loguru import logger
from polars import DataFrame

In [ ]:
logger.remove()
logger.add(sys.stdout, level="INFO")

In [ ]:
with open('src/embedding_tuning/conf.yaml') as file:
    conf = yaml.safe_load(file)

# Prepare data for training

In [ ]:
meals = pl.read_parquet(conf['PATHS']['meals'])


meals = (
    meals
    .select('meal_id', 'embedding', 'meal_type')
    .unique()
)

meals.head()

In [ ]:
pairs = meals.select('meal_id', 'meal_type')
pairs = (
    pairs
    .join(pairs, how='cross')
    .filter(
        (1 == 1)
        & (pl.col('meal_id') != pl.col('meal_id_right'))
        & (pl.col('meal_type') == pl.col('meal_type_right'))
    )
    .select('meal_id', pl.col('meal_id_right').alias('meal_id_pos'))
)

pairs.head()

In [ ]:
map_mealtype2id = {
    "meat": 0,
    "fish": 1,
    "chicken": 2,
    "vegan": 3,
    "vegetarian": 4,
}

class DataEmbedding(Dataset):
    def __init__(self, meals: DataFrame, pairs: DataFrame) -> None:
        super().__init__()

        self.meals = meals
        self.pairs = pairs

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index) -> dict:
        entry = self.pairs[index]
        meal_id, meal_id_pos = entry['meal_id'].item(), entry['meal_id_pos'].item()

        meal_info = self.meals.filter(pl.col('meal_id') == meal_id)
        embd_meal = meal_info['embedding'].item().to_numpy().copy().astype('float32')
        meal_type =meal_info['meal_type'].item()
        
        embd_meal_pos = self.meals.filter(pl.col('meal_id') == meal_id_pos)['embedding'].item().to_numpy().copy().astype('float32')

        meal_type_id = map_mealtype2id[meal_type]

        return {
            'meal': embd_meal,
            'meal_pos': embd_meal_pos,
            'meal_type': meal_type_id
        }
    
# dataset = DataEmbedding(meals, pairs)
# loader = DataLoader(dataset, conf['bsz'], shuffle=True)
# for batch in loader:
#     break

# Prepare model

In [ ]:
class ResNet(Module):
    def __init__(self, d_hid: int) -> None:
        super().__init__()

        self.ff = nn.Sequential(
            nn.LayerNorm(d_hid),
            nn.Linear(d_hid, d_hid),
            nn.ReLU(),
        )

    def forward(self, X: Tensor) -> Tensor:
        return self.ff(X) + X


class EmbeddingLearner(Module):
    def __init__(
        self,
        d_embd: int,
        d_hid: int,
        tau: float = 0.07,
        closeness: float = .9,
    ) -> None:
        super().__init__()

        self.tau = tau
        self.closeness = closeness      # This closeness isn't exact 1. to ensure the learned meal embeddings aren't exactly the same

        d_hid_1 = d_embd // 2
        self.learner = nn.Sequential(
            ResNet(d_embd),
            nn.Linear(d_embd, d_hid_1),
            ResNet(d_hid_1),
            nn.Linear(d_hid_1, d_hid),
        )



    def train(self, meal: Tensor, meal_pos: Tensor, meal_type: Tensor):
        # meal, meal_pos: [bz, d_embd]
        # meal_type: [bz]

        bz, d = meal_pos.shape

        # Encode
        meal = self.forward(meal)
        meal_pos = self.learner(meal_pos)


        # Apply: In-batch negative sampling: Duplicate embedding of positive meals
        meal_pos = meal_pos.unsqueeze(0).repeat(bz, 1, 1)
        # [bz, bz, 1]

        # Calculate similarity
        sim = (meal.unsqueeze(1) @ meal_pos.permute(0, 2, 1)).squeeze(1)
        sim /= self.tau
        # [bz, bz]


        # Create target tensor
        meal_types_others = meal_type.unsqueeze(0).repeat(bz, 1)
        meal_types_current = meal_type.unsqueeze(1).repeat(1, bz)

        tgt = torch.full((bz, bz), -self.closeness, device=meal_pos.device).masked_fill(meal_types_current == meal_types_others, self.closeness)
        # [bz, bz]


        # Calculate loss
        loss = F.mse_loss(sim, tgt)
        # [bz, bz]


        return loss


    def forward(self, meal: Tensor) -> Tensor:
        # meal: [bz, d_embd]

        # Encode
        meal = self.learner(meal)
        # [bz, d_hid]

        # Row-wise L2 normalize
        meal = meal / torch.norm(meal, dim=-1, p=2, keepdim=True)
        # [bz, d_hid]

        logger.debug(f"meal: {meal.shape}")

        return meal

# model = EmbeddingLearner(1024, 32)
# meal = torch.rand((conf['bsz'], 1024))
# meal_pos = torch.rand((conf['bsz'], 1024))
# meal_type = torch.randint(0, 5, (conf['bsz'],))

# sim = model.train(meal, meal_pos, meal_type)
# sim.shape

In [ ]:
from typing import Any


class LitEmbeddingLearner(L.LightningModule):
    def __init__(
        self,
        params: dict,
        lr: float = 3e-4,
    ) -> None:
        super().__init__()
        self.save_hyperparameters()

        self.model = EmbeddingLearner(**params)
        self.lr = lr

    def forward(self, meal: Tensor) -> Any:
        return self.model(meal)

    def training_step(self, batch, batch_idx):
        loss = self.model.train(**batch)

        self.log("train_loss", loss, prog_bar=True, on_step=True)

        return loss

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), lr=self.lr)

        scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.3, total_iters=1000)

        return {"optimizer": optimizer, "lr_scheduler": scheduler}



# Train

In [ ]:
dataset = DataEmbedding(meals, pairs)
loader = DataLoader(dataset, conf['bsz'], shuffle=True)

In [ ]:
params = {
    'd_hid': conf['d_hid'],
    'd_embd': conf['d_embd'],
    'tau': conf['tau'],
}
litmodel = LitEmbeddingLearner(params, float(conf['lr']))

In [ ]:
version = datetime.datetime.now().strftime("%m-%d_%H-%M-%S")
model_name = "embedding-tuning_stage-1"

trainer = L.Trainer(
    # devices=0,
    callbacks=[
        RichProgressBar(leave=True),
        LearningRateMonitor(logging_interval='step')
    ],
    logger=TensorBoardLogger("tb_logs", name=model_name, version=version),
    gradient_clip_val=1,
    max_epochs=conf['num_epoch'],
)

In [ ]:
trainer.fit(litmodel, loader)

## Save checkpoints

In [ ]:
version = datetime.datetime.now().strftime("%m-%d_%H-%M-%S")
path = Path(conf['PATHS']['ckpt'].replace("[date]", version))

path.parent.mkdir(exist_ok=True, parents=True)

trainer.save_checkpoint(path)

# Do inference

In [ ]:
with torch.no_grad():
    encoded = litmodel(torch.tensor(meals['embedding'].to_numpy(), dtype=torch.float32))

(
    meals
    .with_columns(pl.Series(encoded.numpy()).alias('embedding_encoded'))
    .write_parquet("data/processed/embedding_tuning/meal_embds.parquet")
)